In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import time
from collections import defaultdict

def load_graph(path):
    if path.endswith('.csv'):
        try:
            # 1. Try standard fast CSV reading first (handles large files like London perfectly)
            df = pd.read_csv(path, low_memory=False)

            # 2. If it parses the entire row as a single column, it likely uses a non-comma delimiter
            if len(df.columns) < 2:
                # Use the python engine to auto-detect the delimiter (Do NOT use low_memory here!)
                df = pd.read_csv(path, sep=None, engine='python')

            if len(df.columns) < 2:
                raise ValueError("Dataset does not have at least 2 columns to form an edge list.")

            source_col = df.columns[0]
            target_col = df.columns[1]
            df = df.dropna(subset=[source_col, target_col])

            G = nx.from_pandas_edgelist(df, source=source_col, target=target_col)
        except Exception:
            # data=False prevents dictionary conversion errors on extra columns
            G = nx.read_edgelist(path, delimiter=',', data=False, comments='#')
    else:
        G = nx.read_edgelist(path, comments='#')

    # Standardize graph: remove self-loops to prevent array size mismatches later
    G.remove_edges_from(nx.selfloop_edges(G))
    G = nx.convert_node_labels_to_integers(G)
    return G

def sigmoid(x):
    # Clipped to prevent overflow
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

def generate_continuous_encoding(G):
    """
    Generates a random continuous encoding vector 'x' for the network.
    """
    x_dict = {}
    for node in G.nodes():
        # FIX: Use exact neighbor count instead of G.degree to guarantee matching lengths
        deg = len(list(G.neighbors(node)))
        x_dict[node] = np.random.uniform(-10, 10, deg + 1)
    return x_dict

def ANCE(G, x_dict):
    """
    Algorithm 1: Attribute Network Continuous Encoding Method (ANCE)
    """
    L1_edges = []
    L2_edges = []

    # --- Step 1: Continuous to Discrete Mapping ---
    for node in G.nodes():
        neighbors = list(G.neighbors(node))
        d_i = len(neighbors)

        if d_i == 0:
            continue

        x_i = x_dict[node]

        h_i = sigmoid(x_i)
        p_i = softmax(h_i)

        # Exclude the attribute probability (the very last element)
        p_i_edges = p_i[:-1]

        # Find s1: the neighbor with the highest probability
        s1_idx = np.argmax(p_i_edges)
        L1_edges.append((node, neighbors[s1_idx]))

        # Find s2: the neighbor with the second highest probability
        if d_i > 1:
            p_i_masked = p_i_edges.copy()
            p_i_masked[s1_idx] = -1.0
            s2_idx = np.argmax(p_i_masked)
            L2_edges.append((node, neighbors[s2_idx]))
        else:
            L2_edges.append((node, neighbors[s1_idx]))

    # --- Step 2: Decoding L1 and L2 ---
    G1 = nx.Graph()
    G1.add_nodes_from(G.nodes())
    G1.add_edges_from(L1_edges)
    label1 = {node: min(comp) for comp in nx.connected_components(G1) for node in comp}

    G2 = nx.Graph()
    G2.add_nodes_from(G.nodes())
    G2.add_edges_from(L2_edges)
    label2 = {node: min(comp) for comp in nx.connected_components(G2) for node in comp}

    # --- Step 3: Union Operation ---
    G_u = defaultdict(list)
    P_0 = set()

    for node in G.nodes():
        l1, l2 = label1[node], label2[node]

        if l1 == l2:
            G_u[f"Comm_{l1}"].append(node)
        else:
            P_0.add(node)
            G_u[f"Comm_L1_{l1}"].append(node)
            G_u[f"Comm_L2_{l2}"].append(node)

    return list(G_u.values()), P_0, label1

def compute_modularity(G, label_dict):
    """
    Computes modularity using a strict partition (label_dict).
    """
    comm_dict = defaultdict(list)
    for node, c_id in label_dict.items():
        comm_dict[c_id].append(node)

    partition = list(comm_dict.values())

    if len(partition) <= 1:
        return -1.0

    try:
        return nx.algorithms.community.quality.modularity(G, partition)
    except ZeroDivisionError:
        return -1.0

# --- Execution Block ---
datasets = [
    "/content/Dataset_CyberCrime_Sean.csv",
    "/content/london_crime_by_lsoa.csv",
    "/content/twitter_combined.txt.gz",
    "/content/Meetings.csv",
    "/content/Phone_Calls.csv",
    "/content/Roles.csv",
    "/content/Sicilian Mafia.csv",
    "/content/facebook_combined.txt.gz"
]

# Updated formatting to include Comms
print(f"{'Dataset':<32} | {'Nodes':<6} | {'Edges':<7} | {'Comms':<6} | {'Modularity':<10} | {'Runtime (s)':<12}")
print("-" * 88)

for ds in datasets:
    try:
        start_time = time.time()

        G = load_graph(ds)
        nodes_count = G.number_of_nodes()
        edges_count = G.number_of_edges()

        # Skip running the algorithm if the graph is completely empty
        if nodes_count == 0 or edges_count == 0:
            print(f"{ds.split('/')[-1][:30]:<32} | {nodes_count:<6} | {edges_count:<7} | {'-':<6} | {'Graph Empty':<10} | {'N/A':<12}")
            continue

        # 1. Initialize random continuous vector bounded [-10, 10]
        x_dict = generate_continuous_encoding(G)

        # 2. Execute Algorithm 1 (ANCE)
        communities, overlapping_nodes, label1 = ANCE(G, x_dict)

        # Calculate number of communities from the primary L1 partition
        num_communities = len(set(label1.values()))

        # 3. Compute Modularity using the primary L1 partition
        modularity = compute_modularity(G, label1)

        runtime = time.time() - start_time
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {nodes_count:<6} | {edges_count:<7} | {num_communities:<6} | {modularity:<10.4f} | {runtime:<12.4f}")

    except FileNotFoundError:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {'N/A':<10} | {'N/A':<12}")
    except Exception as e:
        ds_name = ds.split('/')[-1][:30]
        print(f"{ds_name:<32} | {'-':<6} | {'-':<7} | {'-':<6} | {str(e)[:10]:<10} | {'N/A':<12}")

Dataset                          | Nodes  | Edges   | Comms  | Modularity | Runtime (s) 
----------------------------------------------------------------------------------------
Dataset_CyberCrime_Sean.csv      | 134    | 160     | 20     | 0.5344     | 0.0549      
london_crime_by_lsoa.csv         | 4868   | 4835    | 33     | 0.9676     | 46.6785     
twitter_combined.txt.gz          | 81306  | 1342296 | 1851   | 0.2408     | 29.3789     
Meetings.csv                     | 95     | 247     | 12     | 0.4105     | 0.0772      
Phone_Calls.csv                  | 92     | 119     | 10     | 0.5622     | 0.0128      
Roles.csv                        | 106    | 80      | 26     | 0.8222     | 0.0138      
Sicilian Mafia.csv               | 143    | 325     | 16     | 0.4420     | 0.0175      
facebook_combined.txt.gz         | 4039   | 88234   | 124    | 0.4264     | 2.2141      
